# فصل نهم: فایل‌ها، مسیرها و تبادل داده
هر مثال مستقل است و فایل‌های تمرینی فقط در پوشهٔ موقت ساخته می‌شوند. خروج از with مربوط به پوشهٔ موقت، دادهٔ آن را پاک می‌کند. برای نگهداری خروجی، راهنمای README را ببینید.

## 09-01 — ساخت مسیر بدون چسباندن رشته‌ها
عملگر / در اینجا مسیر می‌سازد، نه تقسیم عددی. name نام فایل با پسوند، stem نام بدون آخرین پسوند و suffix آخرین پسوند است. ساخت Path نه فایل می‌سازد و نه وجود آن را تأیید می‌کند. as_posix فقط شکل نمایشیِ جداکننده را یکدست می‌کند؛ فایل را جابه‌جا نمی‌کند.

In [ ]:
from pathlib import Path
path = Path("data") / "raw" / "orders.csv"
print(path.as_posix())
print(path.name, path.stem, path.suffix)
print(path.parent.as_posix())


## 09-02 — مسیر نسبی و پوشهٔ کاری
Path.cwd پوشهٔ کاری فرایند را می‌دهد، نه لزوماً پوشهٔ کد. مسیر نسبی هنگام دسترسی به فایل نسبت به همین پوشه تفسیر می‌شود. برای پرهیز از خروجی وابسته به دستگاه، فقط ویژگی مسیر را چاپ کرده‌ایم. تغییر پوشهٔ کاری می‌تواند مقصد همان رشتهٔ نسبی را عوض کند.

In [ ]:
from pathlib import Path
print(Path.cwd().is_absolute())
print(Path("data/orders.csv").is_absolute())


## 09-03 — دیدن نحو Windows بدون وابستگی به سیستم‌عامل
PureWindowsPath فقط نحو مسیر Windows را تحلیل می‌کند و به دیسک دست نمی‌زند؛ روی Linux هم برای این مثال مناسب است. این با ادعای وجود درایو C فرق دارد. رشتهٔ خام تفسیر بسیاری از بک‌اسلش‌ها به‌عنوان escape را متوقف می‌کند؛ رشتهٔ خام نمی‌تواند با یک بک‌اسلش تنها پایان یابد.

In [ ]:
from pathlib import PureWindowsPath
path = PureWindowsPath(r"C:\book\data\orders.csv")
print(path.name)
print(path.drive)
print(path.as_posix())


## 09-04 — ساخت پوشهٔ خروجی در فضای موقت
parents=True والدهای گمشده را می‌سازد. exist_ok=True وجود همان پوشه را قابل‌قبول می‌کند، نه هر نوع مانع یا خطا را. TemporaryDirectory پس از پایان with پاک می‌شود؛ این قرارداد پاک‌سازیِ پوشهٔ موقت است و ویژگی عمومی with نیست. اگر در مسیر موردنظر یک فایل عادی باشد، mkdir همچنان می‌تواند خطا بدهد.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    output = Path(folder) / "outputs" / "daily"
    output.mkdir(parents=True, exist_ok=True)
    print(output.exists(), output.is_dir(), output.is_file())
print(output.exists())


## 09-05 — پیمایش با ترتیب صریح
به ترتیب طبیعیِ glob تکیه نمی‌کنیم؛ نام‌ها را صریح مرتب کرده‌ایم. is_file مانع ورود پوشه‌ای با نام دارای پسوند csv به این فهرست می‌شود. تطبیق بزرگی و کوچکی حروف و بعضی جزئیات فایل‌سیستم ممکن است میان سامانه‌ها متفاوت باشد؛ این دادهٔ کوچک فقط نام‌های ساده دارد.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    root = Path(folder)
    for name in ["b.csv", "a.csv", "notes.txt"]:
        (root / name).write_text("sample", encoding="utf-8")
    names = sorted(path.name for path in root.glob("*.csv")
                   if path.is_file())
    print(names)


## 09-06 — روش کوتاه برای فایل متنی کوچک
write_text فایل را می‌نویسد و می‌بندد؛ اگر فایل قبلاً وجود داشته باشد بازنویسی می‌شود. عدد برگشتی تعداد نویسه‌های متن نوشته‌شده است، نه الزاماً تعداد بایت‌های دیسک. read_text همهٔ فایل را در حافظه می‌خواند. در این مثال ورودی ASCII است و شمار نویسه‌ها و بایت‌های UTF-8 برابر می‌شود؛ این برابری عمومی نیست.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    count = path.write_text("alpha\nbeta\n", encoding="utf-8")
    print(count)
    print(repr(path.read_text(encoding="utf-8")))


## 09-07 — بسته‌شدن فایل پس از بلوک with
با خروج از with مربوط به فایل، close فراخوانی می‌شود؛ حتی اگر اجرای بدنه با استثنا قطع شود. نام target هنوز وجود دارد اما فایل بسته است. بستن فایل و پاک‌کردن آن دو کار جدا هستند. with برای فایل معمولی داده را حذف نمی‌کند و تراکنش یا تضمین دوام در قطع برق هم نمی‌سازد.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    with path.open("x", encoding="utf-8") as target:
        target.write("ready")
        print(target.closed)
    print(target.closed)


## 09-08 — تفاوت write و print
write پایان خط اضافه نمی‌کند؛ print به‌طور پیش‌فرض پایان خط دارد. newline="\n" در نوشتن این نمونه، ترجمهٔ پایان خط را کنترل می‌کند. خروجی repr نشانهٔ پایان خط را قابل‌دیدن می‌کند. پارامتر file مسیر چاپ را به فایل می‌برد؛ این چاپ روی صفحه نیست.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    with path.open("x", encoding="utf-8", newline="\n") as target:
        count = target.write("A")
        print("B", file=target)
    print(count)
    print(repr(path.read_text(encoding="utf-8")))


## 09-09 — افزودن به انتهای فایل
حالت a به انتهای فایل می‌افزاید و فایل غایب را می‌سازد. این حالت به‌تنهایی تضمین نمی‌کند که چند نویسندهٔ هم‌زمان، یک گزارش رویدادِ بدون تداخل تولید کنند. اگر این سلول را بدون پوشهٔ موقت روی فایل ثابت تکرار کنیم، رکوردهای تکراری می‌سازیم؛ اجرای دوباره باید آگاهانه طراحی شود.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "events.txt"
    path.write_text("start\n", encoding="utf-8")
    with path.open("a", encoding="utf-8", newline="\n") as target:
        target.write("finish\n")
    print(path.read_text(encoding="utf-8").splitlines())


## 09-10 — ایجاد انحصاری به‌جای بازنویسی ناخواسته
حالت x تنها وقتی فایل وجود ندارد آن را ایجاد می‌کند. وجود قبلیِ فایل با FileExistsError گزارش می‌شود؛ نمونهٔ خطا در بخش خطاهای عمدی آمده است. الگوی «اول exists، سپس w» میان بررسی و نوشتن فاصلهٔ زمانی دارد و جانشین ایجاد انحصاری نیست. x هم مانع همهٔ خطرهای مجوز، لینک نمادین یا خرابی دستگاه نیست.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "result.txt"
    with path.open("x", encoding="utf-8") as target:
        target.write("approved")
    print(path.read_text(encoding="utf-8"))


## 09-11 — مکان‌نما و پایان فایل
خواندن، مکان‌نمای جریان را جلو می‌برد؛ خواندنِ دوباره خودکار از ابتدا شروع نمی‌شود. رشتهٔ خالی در این نمونه نشانهٔ رسیدن به پایان فایل است. seek(0) به ابتدای جریان برمی‌گردد. در فایل متنی، عدد tell لزوماً شمارهٔ نویسه یا آفست بایتِ قابل‌محاسبه نیست؛ در این فصل فقط بازگشت به ابتدا را به کار می‌بریم.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "letters.txt"
    path.write_text("ABCD", encoding="utf-8")
    with path.open("r", encoding="utf-8") as source:
        print(source.read(2))
        print(source.read())
        print(repr(source.read()))
        source.seek(0)
        print(source.read(1))


## 09-12 — پیمایش خطی بدون حذف فاصله‌های معنادار
در این نمونه فقط علامت پایان خط حذف شده است؛ strip بدون آرگومان، فاصله‌های ابتدا و انتهای داده را هم می‌زداید. خط خالیِ داخل فایل همچنان یک خط است. پردازش خط‌به‌خط، نیاز به نگهداری همهٔ خطوط را کم می‌کند، اما یک خط بسیار بزرگ یا انباشتن نتایج در فهرست هنوز می‌تواند حافظهٔ زیاد بخواهد.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "lines.txt"
    path.write_text("  A  \n\nB\n", encoding="utf-8")
    with path.open("r", encoding="utf-8") as source:
        for number, line in enumerate(source, start=1):
            print(number, repr(line.rstrip("\n")))


## 09-13 — تفاوت خط خالی با پایان فایل
خواندن یک خطِ خالی معمولاً رشتهٔ شامل پایان خط می‌دهد؛ پایان فایل رشتهٔ خالی می‌دهد. اگر پیش از بررسی پایان فایل، بی‌محابا strip کنیم، این دو حالت را یکسان کرده‌ایم. در یک parser ساده، این اشتباه می‌تواند خواندن را زودتر از موعد متوقف کند.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "lines.txt"
    path.write_text("\n", encoding="utf-8")
    with path.open("r", encoding="utf-8") as source:
        print(repr(source.readline()))
        print(repr(source.readline()))


## 09-14 — پایان خط روی دیسک و در متن
در خواندن متنی با تنظیم پیش‌فرض، پایان‌خط‌های رایج به \n تبدیل می‌شوند. read_bytes بایت‌های واقعی را برمی‌گرداند و چنین ترجمه‌ای ندارد. برابرنبودن بایت‌ها لزوماً به معنی تفاوت دادهٔ متنی نیست؛ برای کنترل یک انتقال باید از ابتدا معلوم باشد برابریِ بایتی می‌خواهیم یا برابری معنایی.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "lines.txt"
    path.write_bytes(b"A\r\nB\r\n")
    print(repr(path.read_bytes()))
    print(repr(path.read_text(encoding="utf-8")))


## 09-15 — نویسه با بایت یکسان نیست
یک نویسهٔ لاتینِ غیر ASCII با escape نوشته شده تا خودِ کد و خروجی همچنان ASCII بمانند. UTF-8 آن را با دو بایت نمایش می‌دهد. encode متن را به بایت و decode بایت را به متن تبدیل می‌کند. شمار نویسهٔ Python هم لزوماً شمار حرفِ قابل‌دیدنِ کاربر نیست؛ ترکیب نشانه‌ها و خوشه‌های نوشتاری در این فصل بررسی نمی‌شود.

In [ ]:
text = "\u00e9"
encoded = text.encode("utf-8")
print(len(text), len(encoded))
print(list(encoded))
print(encoded.decode("utf-8") == text)


## 09-16 — خواندن UTF-8 همراه BOM
BOM آغاز یک فایل UTF-8 ممکن است بی‌سروصدا بخشی از نام اولین ستون شود. utf-8-sig در خواندن، این امضای آغازین را در صورت وجود برمی‌دارد و فایل UTF-8 بدون BOM را هم می‌خواند. این تنظیم، تشخیص‌دهندهٔ همهٔ کدگذاری‌ها نیست. در نوشتن با همین نام کدگذاری، BOM اضافه می‌شود؛ پروژهٔ ما خروجی UTF-8 بدون BOM می‌دهد.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "header.txt"
    path.write_bytes(b"\xef\xbb\xbforder_id\n")
    plain = path.read_text(encoding="utf-8")
    clean = path.read_text(encoding="utf-8-sig")
    print(plain.startswith("\ufeff"))
    print(clean.splitlines()[0])


## 09-17 — CSV و ویرگول داخل مقدار
نقل‌قول‌ها باعث می‌شوند ویرگولِ داخل category جداکنندهٔ ستون محسوب نشود. csv.reader به‌طور پیش‌فرض فهرستی از رشته‌ها برای هر رکورد می‌دهد؛ نه تاپل و نه تبدیل خودکار عدد. StringIO یک جریان متنیِ درون حافظه است و اینجا صرفاً نمایش کوتاه‌ترِ قواعد قالب را ممکن می‌کند.

In [ ]:
import csv
from io import StringIO
text = 'order_id,category,quantity\nA01,"Books, Stationery",2\n'
with StringIO(text, newline="") as source:
    rows = list(csv.reader(source))
print(rows[1])


## 09-18 — چرا split جای parser نیست؟
این خروجی چهار تکه دارد، در حالی که رکورد سه فیلد دارد. split از نقل‌قول CSV، نقل‌قولِ دوتایی و پایان خطِ داخل فیلد خبر ندارد. تغییر جداکننده به سمی‌کالن نیز مسئلهٔ عمومی را حل نمی‌کند؛ قالب و parser باید با هم قرارداد داشته باشند.

In [ ]:
line = 'A01,"Books, Stationery",2'
print(line.split(","))


## 09-19 — DictReader نوع ستون را حدس نمی‌زند
شناسهٔ 0012 باید متن بماند تا صفرهای آغازین از بین نرود. quantity هنوز رشته است؛ همین ظاهر عددی به آن معنای عددی نمی‌دهد. فرمت فایل CSV به‌تنهایی فرهنگ دادهٔ ستون‌ها را حمل نمی‌کند. واحد پول، نوع شناسه و دامنهٔ مجاز باید جدا مستند شوند.

In [ ]:
import csv
from io import StringIO
text = "order_id,quantity\n0012,3\n"
with StringIO(text, newline="") as source:
    row = next(csv.DictReader(source))
print(row["order_id"], type(row["order_id"]).__name__)
print(row["quantity"], type(row["quantity"]).__name__)


## 09-20 — تبدیل فقط ستون‌های عددی
تبدیل نوع انتخابی است، نه اجرای int روی همهٔ فیلدها. int روی دادهٔ گمشده یا متن نامعتبر خطا می‌دهد. موفقیت تبدیل، مثبت‌بودن تعداد یا معقول‌بودن قیمت را تضمین نمی‌کند. باید دریافت بایت، تجزیهٔ قالب، تبدیل نوع، بررسی قرارداد و سپس محاسبه را مراحل جدا دانست.

In [ ]:
import csv
from io import StringIO
text = "order_id,quantity,unit_price_toman\n0012,3,100\n"
with StringIO(text, newline="") as source:
    row = next(csv.DictReader(source))
for field in ["quantity", "unit_price_toman"]:
    row[field] = int(row[field])
print(row["order_id"])
print(row["quantity"] * row["unit_price_toman"])


## 09-21 — نوشتن CSV و بازخوانی مقدارهای ویژه
دو رکورد داده به‌همراه سرستون، به‌علت پایان خط داخل یک فیلد، چهار خط فیزیکی ساخته‌اند. تعداد خطوط منهای یک، روش قابل‌اعتماد شمارش رکورد CSV نیست. newline="" را هم در خواندن و هم نوشتن می‌دهیم تا ماژول csv قواعد پایان خط را مدیریت کند. writer نقل‌قول‌گذاری را انجام می‌دهد؛ خودمان join نمی‌کنیم.

In [ ]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory
rows = [{"order_id": "A01", "category": 'Tools, "Small"'},
        {"order_id": "A02", "category": "Line1\nLine2"}]
with TemporaryDirectory() as folder:
    path = Path(folder) / "items.csv"
    with path.open("x", encoding="utf-8", newline="") as target:
        writer = csv.DictWriter(target, fieldnames=["order_id", "category"],
                                lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    with path.open("r", encoding="utf-8", newline="") as source:
        restored = list(csv.DictReader(source))
    print(restored == rows)
    print(len(restored), len(path.read_text(encoding="utf-8").splitlines()))


## 09-22 — جداکنندهٔ قراردادی
پسوند csv تضمین نمی‌کند جداکننده ویرگول باشد. جداکننده را از قرارداد تولیدکننده می‌گیریم؛ حدس‌زدن خودکار روی نمونهٔ کوچک ممکن است نادرست باشد. تغییر delimiter به‌تنهایی فرمت عدد اعشاریِ محلی یا جداکنندهٔ هزارگان را تفسیر نمی‌کند.

In [ ]:
import csv
from io import StringIO
text = "order_id;quantity\nA01;2\n"
with StringIO(text, newline="") as source:
    row = next(csv.DictReader(source, delimiter=";"))
print(row["order_id"], int(row["quantity"]))


## 09-23 — فیلد خالی با فیلد غایب فرق دارد
در رکورد اول فیلد وجود دارد اما مقدارش رشتهٔ خالی است. در رکورد دوم تعداد فیلدها کم است و DictReader مقدار None می‌گذارد. اگر فیلد اضافه باشد، در تنظیم پیش‌فرض زیر کلید None جمع می‌شود. این رفتارها خودکار schema را اعتبارسنجی نمی‌کنند؛ پروژه هر دو ناهمخوانی را صریح رد می‌کند. برای مقدار گمشده، جانشانیِ بی‌اجازه با صفر ممنوع است.

In [ ]:
import csv
from io import StringIO
text = "order_id,quantity\nA01,\nA02\n"
with StringIO(text, newline="") as source:
    rows = list(csv.DictReader(source))
print(repr(rows[0]["quantity"]))
print(rows[1]["quantity"] is None)


## 09-24 — JSON متن است، نه دیکشنری Python
json.loads متن را به شیء Python تبدیل می‌کند. true و null در JSON به True و None در Python تبدیل می‌شوند؛ این دو زبان را نباید با هم مخلوط کرد. نقل‌قول تک، کامنت و ویرگولِ انتهایی جزو نحو استاندارد JSON نیستند. JSON معتبر هم لزوماً یک سفارش معتبر نیست.

In [ ]:
import json
text = '{"quantity": 2, "approved": true, "note": null}'
row = json.loads(text)
print(row["quantity"], type(row["quantity"]).__name__)
print(row["approved"], row["note"])


## 09-25 — تبدیل شیء به متن JSON
dumps یک رشته می‌سازد و خودش فایل نمی‌نویسد. sort_keys فقط برای نمایش منظم کلیدهاست و ربطی به مرتب‌کردن رکوردها ندارد. ensure_ascii نویسه‌های غیر ASCII را در نمایش JSON escape می‌کند؛ مترجم زبان نیست و انتخاب کدگذاری فایل را هم به عهده نمی‌گیرد. allow_nan=False از تولید مقادیر غیر استاندارد NaN و Infinity جلوگیری می‌کند.

In [ ]:
import json
row = {"order_id": "0012", "quantity": 2, "note": None}
text = json.dumps(row, ensure_ascii=True, sort_keys=True,
                  allow_nan=False)
print(text)
print(json.loads(text) == row)


## 09-26 — load و dump برای جریان فایل
load و dump با جریان کار می‌کنند؛ loads و dumps با متن. پسوند s در این دو نام یادآور string است. بازخوانی این مثال نشان می‌دهد مقدارها برای schema کوچک حاضر حفظ شده‌اند؛ برابری همهٔ انواع Python در رفت‌وبرگشت JSON از این مثال نتیجه نمی‌شود.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
rows = [{"order_id": "A01", "quantity": 2}]
with TemporaryDirectory() as folder:
    path = Path(folder) / "orders.json"
    with path.open("x", encoding="utf-8", newline="\n") as target:
        json.dump(rows, target, indent=2, ensure_ascii=True,
                  allow_nan=False)
        target.write("\n")
    with path.open("r", encoding="utf-8") as source:
        restored = json.load(source)
    print(restored == rows)


## 09-27 — رفت‌وبرگشت JSON همیشه بی‌اتلاف نیست
کلید عددی به رشته و تاپل به فهرست تبدیل شده است. JSON قالب نگهداری عینِ همهٔ اشیای Python نیست. مجموعه، شیء تاریخ یا یک شیء سفارشی نیز بدون تبدیل قراردادی به JSON معمولی تبدیل نمی‌شود. پروژهٔ ما شناسه و تاریخ را متن و شمارش‌ها را int نگه می‌دارد تا این ابهام‌ها محدود شوند.

In [ ]:
import json
original = {1: ("A", "B")}
restored = json.loads(json.dumps(original))
print(restored)
print(restored == original)


## 09-28 — نام تکراری در شیء JSON
در تنظیم پیش‌فرضِ ماژول json، آخرین مقدارِ نام تکراری باقی می‌ماند. این تکرار با دو سفارش دارای order_id برابر فرق دارد: اولی درون یک شیء JSON است و ممکن است قبل از رسیدن به validator از بین رفته باشد؛ دومی میان رکوردهاست. نسخهٔ آموزشی پروژه تکرار نام‌های درون JSON را کشف نمی‌کند؛ برای ورودی حساس باید parser و سیاست جدا طراحی شود.

In [ ]:
import json
row = json.loads('{"quantity": 2, "quantity": 9}')
print(row["quantity"])


## 09-29 — برابری معناییِ CSV و JSON پس از تبدیل
پیش از تبدیل نوع، مقدار "2" با عدد 2 برابر نیست. پس از تبدیلِ قراردادی، دو نمایش به یک داده می‌رسند. برابری فهرست‌ها ترتیب رکوردها را هم می‌سنجد؛ برابری دیکشنری‌ها به ترتیب کلیدها وابسته نیست. اگر ترتیب رکوردها بخشی از قرارداد نباشد، باید مقایسه با کلید یکتا و کنترل تکرار جدا تعریف شود.

In [ ]:
import csv
import json
from io import StringIO
csv_text = "order_id,quantity\n0012,2\n"
json_text = '[{"order_id": "0012", "quantity": 2}]'
with StringIO(csv_text, newline="") as source:
    csv_rows = list(csv.DictReader(source))
json_rows = json.loads(json_text)
print(csv_rows == json_rows)
csv_rows[0]["quantity"] = int(csv_rows[0]["quantity"])
print(csv_rows == json_rows)


## 09-30 — کارگاه کوچکِ مستقل: فایل تا جمع مبلغ
این مثال دو رکوردِ کوچک و متفاوت از دادهٔ اصلی دارد تا جریان فایل تا نتیجه یکجا دیده شود؛ انتظار آن ۳۵۰، ۱۰۰ و ۲۵۰ تومان است. ورودیِ این مثال از قبل معتبر فرض شده است. آن را با پروژهٔ کاملِ ۱۲ سفارش و validator بعدی اشتباه نگیرید. مبلغ پس از مرجوعی هنوز سود نیست و مقایسهٔ شیء نتیجه جای آزمون مستقل محاسبه را نمی‌گیرد.

In [ ]:
import csv
import json
from pathlib import Path
from tempfile import TemporaryDirectory
rows = [
    {"order_id": "0012", "quantity": 2,
     "unit_price_toman": 100, "returned_quantity": 1},
    {"order_id": "0013", "quantity": 3,
     "unit_price_toman": 50, "returned_quantity": 0},
]
with TemporaryDirectory() as folder:
    root = Path(folder)
    path = root / "orders.csv"
    with path.open("x", encoding="utf-8", newline="") as target:
        writer = csv.DictWriter(target, fieldnames=list(rows[0]),
                                lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    with path.open("r", encoding="utf-8", newline="") as source:
        loaded = list(csv.DictReader(source))
    for row in loaded:
        for field in ["quantity", "unit_price_toman", "returned_quantity"]:
            row[field] = int(row[field])
    gross = sum(row["quantity"] * row["unit_price_toman"]
                for row in loaded)
    refund = sum(row["returned_quantity"] * row["unit_price_toman"]
                 for row in loaded)
    result = {"gross_toman": gross, "refund_toman": refund,
              "retained_toman": gross - refund}
    output = root / "report.json"
    with output.open("x", encoding="utf-8") as target:
        json.dump(result, target, allow_nan=False)
    with output.open("r", encoding="utf-8") as source:
        restored = json.load(source)
    print(gross, refund, gross - refund)
    print(restored == result)
